# Calibration Confidence

Ce notebook reprend le script `calibration_confidence.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Calibre les probabilites/seuils qui seront utilises dans une decision live.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Calibration and bootstrap confidence intervals for risk predictions.
- Artefacts controles : Sequence calibration/bootstrap exists. (`runs/exp_010_sequence_len60_focal_catalogue/calibration_tcn_aug/test_bootstrap_ci.csv`); Fusion calibration/bootstrap exists. (`runs/exp_014_sequence_crop_fusion/calibration_tcn_aug_meta/test_bootstrap_ci.csv`).

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "calibration_confidence.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
import math
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, brier_score_loss, precision_recall_fscore_support, roc_auc_score

from ml_pipeline import ROOT, alarm_episodes, safe_auc, threshold_sweep, write_json


## Fonction `ece_score`

Cette cellule definit `ece_score`. Elle prepare une partie du script.

In [ ]:
def ece_score(y, p, n_bins=10):
    y = np.asarray(y, dtype=np.float32)
    p = np.asarray(p, dtype=np.float32)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    rows = []
    ece = 0.0
    for i in range(n_bins):
        left, right = edges[i], edges[i + 1]
        mask = (p >= left) & (p < right if i < n_bins - 1 else p <= right)
        if not mask.any():
            rows.append({"bin": i, "left": left, "right": right, "n": 0, "confidence": None, "empirical": None})
            continue
        conf = float(p[mask].mean())
        empirical = float(y[mask].mean())
        weight = float(mask.mean())
        ece += weight * abs(conf - empirical)
        rows.append({"bin": i, "left": left, "right": right, "n": int(mask.sum()), "confidence": conf, "empirical": empirical})
    return float(ece), rows


## Fonction `metrics_at_threshold`

Cette cellule definit `metrics_at_threshold`. Elle prepare une partie du script.

In [ ]:
def metrics_at_threshold(df, score_col, threshold, label_col):
    y = df[label_col].astype(int).to_numpy()
    p = df[score_col].to_numpy()
    pred = (p >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(y, pred, average="binary", zero_division=0)
    danger_total = 0
    danger_hit = 0
    safe_alarm_count = 0
    safe_minutes = 0.0
    for _, group in df.groupby("video_id"):
        group = group.sort_values("time_s")
        is_danger = int(group["is_danger_clip"].iloc[0]) if "is_danger_clip" in group else int(group[label_col].max())
        alarms = alarm_episodes(group["time_s"], group[score_col], threshold, gap_s=1.0, persistence_windows=2)
        if is_danger:
            danger_total += 1
            target_values = group["target_time_s"].replace("", np.nan).astype(float).dropna() if "target_time_s" in group else pd.Series([], dtype=float)
            target = float(target_values.iloc[0]) if len(target_values) else math.nan
            if math.isnan(target):
                if alarms:
                    danger_hit += 1
            else:
                useful = [alarm for alarm in alarms if alarm <= target + 0.5]
                if useful:
                    danger_hit += 1
        else:
            safe_alarm_count += len(alarms)
            safe_minutes += max(1e-6, float(group["time_s"].max()) / 60.0)
    return {
        "n_windows": int(len(df)),
        "positive_windows": int(y.sum()),
        "average_precision": safe_auc(average_precision_score, y, p),
        "roc_auc": safe_auc(roc_auc_score, y, p),
        "brier": float(brier_score_loss(y, p)) if len(np.unique(y)) > 1 else None,
        "window_precision": float(precision),
        "window_recall": float(recall),
        "window_f1": float(f1),
        "danger_clip_hit_rate": float(danger_hit / danger_total) if danger_total else None,
        "safe_false_alarms_per_min": float(safe_alarm_count / safe_minutes) if safe_minutes > 0 else 0.0,
        "danger_clip_total": int(danger_total),
        "safe_alarm_count": int(safe_alarm_count),
    }


## Fonction `choose_threshold`

Cette cellule definit `choose_threshold`. Elle prepare une partie du script.

In [ ]:
def choose_threshold(df, score_col, label_col):
    if label_col != "danger_within_1.0s":
        candidates = []
        for threshold in np.arange(0.05, 1.0, 0.05):
            row = metrics_at_threshold(df, score_col, float(threshold), label_col)
            row["threshold"] = float(threshold)
            candidates.append(row)
        sweep = pd.DataFrame(candidates)
    else:
        sweep = threshold_sweep(df.rename(columns={score_col: "risk"}), "risk", 1.0, "val", persistence_windows=2)
    best = sweep.sort_values(["danger_clip_hit_rate", "safe_false_alarms_per_min", "window_precision"], ascending=[False, True, False]).iloc[0]
    return float(best["threshold"]), sweep


## Fonction `bootstrap_ci`

Cette cellule definit `bootstrap_ci`. Elle prepare une partie du script.

In [ ]:
def bootstrap_ci(df, score_col, threshold, label_col, n_boot, seed):
    rng = np.random.default_rng(seed)
    videos = df["video_id"].drop_duplicates().to_numpy()
    rows = []
    groups = {video_id: group for video_id, group in df.groupby("video_id")}
    for _ in range(n_boot):
        sampled = rng.choice(videos, size=len(videos), replace=True)
        boot = pd.concat([groups[video_id] for video_id in sampled], ignore_index=True)
        rows.append(metrics_at_threshold(boot, score_col, threshold, label_col))
    boot = pd.DataFrame(rows)
    ci_rows = []
    for col in ["average_precision", "roc_auc", "brier", "window_precision", "window_recall", "window_f1", "danger_clip_hit_rate", "safe_false_alarms_per_min"]:
        values = pd.to_numeric(boot[col], errors="coerce").dropna().to_numpy()
        if len(values) == 0:
            continue
        ci_rows.append(
            {
                "metric": col,
                "mean": float(np.mean(values)),
                "std": float(np.std(values, ddof=0)),
                "ci_2.5": float(np.quantile(values, 0.025)),
                "ci_50": float(np.quantile(values, 0.50)),
                "ci_97.5": float(np.quantile(values, 0.975)),
            }
        )
    return pd.DataFrame(ci_rows), boot


## Fonction `make_calibration_plot`

Cette cellule definit `make_calibration_plot`. Elle prepare une partie du script.

In [ ]:
def make_calibration_plot(out_dir, split_name, rows):
    df = pd.DataFrame(rows)
    df = df[df["n"] > 0].copy()
    if df.empty:
        return
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)
    ax.scatter(df["confidence"], df["empirical"], s=np.maximum(20, df["n"].to_numpy() * 1.5), alpha=0.75)
    ax.set_xlabel("mean predicted risk")
    ax.set_ylabel("empirical positive rate")
    ax.set_title(f"Calibration: {split_name}")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    fig.tight_layout()
    fig.savefig(out_dir / f"calibration_{split_name}.png", dpi=160)
    plt.close(fig)


## Fonction `run`

Cette cellule definit `run`. Elle prepare une partie du script.

In [ ]:
def run(args):
    input_path = Path(args.input)
    if not input_path.is_absolute():
        input_path = ROOT / input_path
    out_dir = Path(args.out_dir)
    if not out_dir.is_absolute():
        out_dir = ROOT / out_dir
    out_dir.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(input_path)
    val = df[df["split"] == "val"].copy()
    test = df[df["split"] == "test"].copy()
    threshold, val_sweep = choose_threshold(val, args.score_col, args.label_col)
    val_sweep.to_csv(out_dir / "validation_threshold_sweep.csv", index=False)
    metric_rows = []
    calibration_rows = []
    for split_name, split_df in [("train", df[df["split"] == "train"].copy()), ("val", val), ("test", test)]:
        metrics = metrics_at_threshold(split_df, args.score_col, threshold, args.label_col)
        ece, bins = ece_score(split_df[args.label_col].astype(int), split_df[args.score_col], args.n_bins)
        metrics.update({"split": split_name, "threshold": threshold, "ece": ece})
        metric_rows.append(metrics)
        for row in bins:
            row.update({"split": split_name})
        calibration_rows.extend(bins)
        make_calibration_plot(out_dir, split_name, bins)
    pd.DataFrame(metric_rows).to_csv(out_dir / "calibration_metrics.csv", index=False)
    pd.DataFrame(calibration_rows).to_csv(out_dir / "calibration_bins.csv", index=False)
    ci, boot = bootstrap_ci(test, args.score_col, threshold, args.label_col, args.bootstrap, args.seed)
    ci.to_csv(out_dir / "test_bootstrap_ci.csv", index=False)
    boot.to_csv(out_dir / "test_bootstrap_samples.csv", index=False)
    write_json(
        out_dir / "calibration_config.json",
        {
            "input": str(input_path),
            "score_col": args.score_col,
            "label_col": args.label_col,
            "threshold_chosen_on_validation": threshold,
            "bootstrap": args.bootstrap,
            "bootstrap_unit": "parent video",
        },
    )
    lines = ["# Calibration And Threshold Confidence", ""]
    lines.append(f"Input: `{input_path}`")
    lines.append(f"Score: `{args.score_col}`")
    lines.append(f"Threshold selected on validation: `{threshold:.3f}`")
    lines.append("")
    lines.append("| split | AP | ROC AUC | Brier | ECE | hit | FA/min | window precision | window recall |")
    lines.append("|---|---:|---:|---:|---:|---:|---:|---:|---:|")
    for row in metric_rows:
        lines.append(
            f"| {row['split']} | {row['average_precision'] if row['average_precision'] is not None else 'NA'} | {row['roc_auc'] if row['roc_auc'] is not None else 'NA'} | {row['brier'] if row['brier'] is not None else 'NA'} | {row['ece']:.3f} | {row['danger_clip_hit_rate'] if row['danger_clip_hit_rate'] is not None else 'NA'} | {row['safe_false_alarms_per_min']:.3f} | {row['window_precision']:.3f} | {row['window_recall']:.3f} |"
        )
    lines.append("")
    lines.append("## Test Bootstrap 95% Intervals")
    lines.append("")
    lines.append("| metric | mean | 2.5% | median | 97.5% |")
    lines.append("|---|---:|---:|---:|---:|")
    for _, row in ci.iterrows():
        lines.append(f"| {row['metric']} | {row['mean']:.3f} | {row['ci_2.5']:.3f} | {row['ci_50']:.3f} | {row['ci_97.5']:.3f} |")
    (out_dir / "calibration_confidence_summary.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
    print(out_dir)


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Calibration and bootstrap confidence intervals for risk predictions.")
    parser.add_argument("--input", required=True)
    parser.add_argument("--score-col", required=True)
    parser.add_argument("--out-dir", required=True)
    parser.add_argument("--label-col", default="danger_within_1.0s")
    parser.add_argument("--bootstrap", type=int, default=500)
    parser.add_argument("--seed", type=int, default=123)
    parser.add_argument("--n-bins", type=int, default=10)
    args = parser.parse_args()
    run(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement concret de l'audit de calibration
# Cette configuration reproduit la calibration du modele sequence TCN focal.
from datetime import datetime
import sys

OUT_DIR = f"runs/exp_010_sequence_len60_focal_catalogue/calibration_tcn_aug_notebook_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
NOTEBOOK_ARGS = [
    "--input", "runs/exp_010_sequence_len60_focal_catalogue/features/predictions_tcn_aug.csv",
    "--score-col", "prob_1.0s",
    "--out-dir", OUT_DIR,
]

ancien_argv = sys.argv[:]
sys.argv = ["calibration_confidence.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
